# Preparación y Procesamiento de Datos

## Proyecto

**Robot Educativo basado en Visión Artificial para apoyar la Comprensión Lectora en Educación General Básica**

**Estudiantes:** Julio Encalada y Sara Cruz  

**Modelo de visión artificial:** YOLOv8  

---

## Objetivo del notebook

El presente notebook tiene como finalidad diseñar e implementar la fase de preparación y procesamiento del conjunto de datos utilizado para el entrenamiento del modelo de visión artificial basado en YOLO. Esta etapa permite organizar, limpiar, verificar y transformar el dataset, asegurando que las imágenes y anotaciones se encuentren en condiciones adecuadas para el proceso de modelado.

A partir de los resultados obtenidos en el análisis exploratorio de datos (EDA), se desarrolla un flujo de trabajo orientado a la validación de la estructura del dataset, revisión de anotaciones, identificación de inconsistencias, estrategias de balanceamiento, aplicación de técnicas de aumento de datos y preparación final del conjunto de imágenes para el entrenamiento del modelo.

In [ ]:
# ============================================================
# 1. PREPARACIÓN Y PROCESAMIENTO DE DATOS
# Robot Educativo - Dataset YOLO
# ============================================================

# Librerías del sistema
import os
import zipfile
from pathlib import Path

# Librerías para manejo de datos
import pandas as pd
import numpy as np

# Librerías para imágenes
import cv2
from PIL import Image

# Librerías para gráficos
import matplotlib.pyplot as plt

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [ ]:
# ============================================================
# 2. CARGA DEL DATASET
# ------------------------------------------------------------
# Esta celda permite seleccionar y cargar el archivo ZIP
# exportado desde Roboflow, el cual contiene el conjunto de
# datos en formato YOLO que será utilizado durante las etapas
# de preparación y procesamiento.
# ============================================================

from google.colab import files

print("Seleccione el archivo ZIP del dataset exportado desde Roboflow.")

uploaded = files.upload()

Seleccione el archivo ZIP del dataset exportado desde Roboflow.


Saving DatasetRobot.v2i.yolov8.zip to DatasetRobot.v2i.yolov8.zip


In [ ]:
# ============================================================
# 3. EXTRACCIÓN DEL DATASET
# ------------------------------------------------------------
# Esta celda descomprime el archivo ZIP seleccionado y
# almacena su contenido en un directorio de trabajo dentro
# del entorno de Google Colab para su posterior procesamiento.
# ============================================================

import zipfile
import os

# Obtener el nombre del archivo cargado
zip_file = list(uploaded.keys())[0]

# Carpeta destino
extract_path = "dataset"

# Extraer el contenido
with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extraído correctamente.")
print(f"Directorio de trabajo: {extract_path}")

Dataset extraído correctamente.
Directorio de trabajo: dataset


# 4. Verificación de la estructura del dataset

En esta etapa se verifica que el conjunto de datos cumpla con la organización requerida por el modelo YOLO para el entrenamiento. Específicamente, se comprueba la existencia de las carpetas correspondientes a los subconjuntos de entrenamiento (*train*), validación (*valid*) y prueba (*test*), así como de los directorios que almacenan las imágenes y sus respectivas anotaciones.

Esta validación constituye un paso previo al procesamiento del dataset, ya que una estructura incorrecta impediría el correcto entrenamiento del modelo de detección de objetos.

In [ ]:
# ============================================================
# 4. VERIFICACIÓN DE LA ESTRUCTURA DEL DATASET
# ------------------------------------------------------------
# Esta celda verifica que el conjunto de datos extraído
# contenga la estructura estándar requerida por YOLO para
# las carpetas de entrenamiento, validación y prueba.
# ============================================================

# Directorios esperados
required_dirs = [
    "train/images",
    "train/labels",
    "valid/images",
    "valid/labels",
    "test/images",
    "test/labels"
]

print("="*60)
print("VERIFICACIÓN DE LA ESTRUCTURA DEL DATASET")
print("="*60)

all_ok = True

for directory in required_dirs:

    path = os.path.join(extract_path, directory)

    if os.path.isdir(path):
        print(f"✅ {directory}")
    else:
        print(f"❌ {directory}  (No encontrado)")
        all_ok = False

print("\n" + "="*60)

if all_ok:
    print("La estructura del dataset es correcta.")
else:
    print("Se encontraron problemas en la estructura del dataset.")

VERIFICACIÓN DE LA ESTRUCTURA DEL DATASET
✅ train/images
✅ train/labels
✅ valid/images
✅ valid/labels
✅ test/images
✅ test/labels

La estructura del dataset es correcta.


**Interpretación**

La verificación confirmó que el conjunto de datos presenta la estructura estándar requerida por YOLO, organizada en los subconjuntos de entrenamiento, validación y prueba, cada uno con sus respectivas carpetas de imágenes y anotaciones. Esta organización garantiza que el dataset pueda ser utilizado correctamente durante el proceso de entrenamiento y evaluación del modelo de detección de objetos.

# 5. Validación del conjunto de datos

En esta sección se realiza una validación técnica del dataset con el propósito de identificar posibles inconsistencias entre las imágenes y sus anotaciones. Esta etapa permite verificar que cada imagen tenga su archivo de etiqueta correspondiente y que no existan anotaciones aisladas sin imagen asociada.

Este proceso forma parte del pipeline de limpieza de datos, ya que permite detectar errores que podrían afectar el entrenamiento del modelo YOLO.

In [ ]:
# ============================================================
# 5. VALIDACIÓN DEL CONJUNTO DE DATOS
# ------------------------------------------------------------
# Esta celda verifica la correspondencia entre imágenes y
# archivos de etiquetas en los subconjuntos train, valid y test.
# Permite identificar imágenes sin anotación y etiquetas sin
# imagen asociada.
# ============================================================

# Extensiones de imagen válidas
image_extensions = [".jpg", ".jpeg", ".png"]

# Subconjuntos del dataset
subsets = ["train", "valid", "test"]

validation_results = []

print("="*60)
print("VALIDACIÓN DE IMÁGENES Y ETIQUETAS")
print("="*60)

for subset in subsets:
    images_path = Path(extract_path) / subset / "images"
    labels_path = Path(extract_path) / subset / "labels"

    image_files = []
    for ext in image_extensions:
        image_files.extend(images_path.glob(f"*{ext}"))

    label_files = list(labels_path.glob("*.txt"))

    image_names = set([img.stem for img in image_files])
    label_names = set([lbl.stem for lbl in label_files])

    images_without_labels = image_names - label_names
    labels_without_images = label_names - image_names

    validation_results.append({
        "Subconjunto": subset,
        "Imágenes": len(image_files),
        "Etiquetas": len(label_files),
        "Imágenes sin etiqueta": len(images_without_labels),
        "Etiquetas sin imagen": len(labels_without_images)
    })

    print(f"\n{subset.upper()}")
    print(f"Imágenes encontradas: {len(image_files)}")
    print(f"Etiquetas encontradas: {len(label_files)}")
    print(f"Imágenes sin etiqueta: {len(images_without_labels)}")
    print(f"Etiquetas sin imagen: {len(labels_without_images)}")


validation_df = pd.DataFrame(validation_results)

print("\nResumen de validación:")
display(validation_df)



VALIDACIÓN DE IMÁGENES Y ETIQUETAS

TRAIN
Imágenes encontradas: 117
Etiquetas encontradas: 117
Imágenes sin etiqueta: 0
Etiquetas sin imagen: 0

VALID
Imágenes encontradas: 11
Etiquetas encontradas: 11
Imágenes sin etiqueta: 0
Etiquetas sin imagen: 0

TEST
Imágenes encontradas: 6
Etiquetas encontradas: 6
Imágenes sin etiqueta: 0
Etiquetas sin imagen: 0

Resumen de validación:


,Subconjunto,Imágenes,Etiquetas,Imágenes sin etiqueta,Etiquetas sin imagen
0,train,117,117,0,0
1,valid,11,11,0,0
2,test,6,6,0,0


### 5.1 Interpretación de los resultados

La validación del conjunto de datos evidenció que todos los subconjuntos del dataset (entrenamiento, validación y prueba) presentan una correspondencia completa entre las imágenes y sus respectivas anotaciones. No se identificaron imágenes sin archivo de etiqueta ni archivos de anotación sin su imagen asociada, lo que confirma la consistencia estructural del conjunto de datos.

Estos resultados indican que el dataset se encuentra correctamente organizado para ser utilizado durante el entrenamiento del modelo YOLO, reduciendo la posibilidad de errores derivados de inconsistencias en las anotaciones.

# 6. Verificación del formato de las anotaciones YOLO

Las anotaciones en formato YOLO deben cumplir una estructura específica para que el proceso de entrenamiento se realice correctamente. Cada línea de un archivo de etiquetas debe contener cinco valores: el identificador de la clase, las coordenadas normalizadas del centro del objeto (*x_center*, *y_center*) y las dimensiones de la caja delimitadora (*width* y *height*).

En esta sección se verifica que todos los archivos de anotación presenten el formato correcto y que las coordenadas se encuentren dentro del rango permitido entre 0 y 1.

In [ ]:
# ============================================================
# 6. VERIFICACIÓN DEL FORMATO DE LAS ANOTACIONES YOLO
# ------------------------------------------------------------
# Esta celda verifica que los archivos de etiquetas cumplan
# con el formato estándar de YOLO:
# clase x_center y_center width height
# Además, comprueba que las coordenadas estén normalizadas.
# ============================================================

from pathlib import Path

subsets = ["train", "valid", "test"]

total_labels = 0
correct_labels = 0
errors = []

for subset in subsets:

    labels_path = Path(extract_path) / subset / "labels"

    for txt_file in labels_path.glob("*.txt"):

        total_labels += 1

        with open(txt_file, "r") as f:
            lines = f.readlines()

        valid = True

        for line in lines:

            values = line.strip().split()

            # Deben existir exactamente 5 valores
            if len(values) != 5:
                valid = False
                break

            try:

                clase = int(values[0])
                coords = list(map(float, values[1:]))

                # Verificar normalización
                if not all(0 <= c <= 1 for c in coords):
                    valid = False
                    break

            except:
                valid = False
                break

        if valid:
            correct_labels += 1
        else:
            errors.append(txt_file.name)

print("="*60)
print("VERIFICACIÓN DEL FORMATO YOLO")
print("="*60)

print(f"Archivos analizados : {total_labels}")
print(f"Archivos correctos  : {correct_labels}")
print(f"Archivos con errores: {len(errors)}")

if len(errors) == 0:
    print("\n✅ Todas las anotaciones cumplen el formato YOLO.")
else:
    print("\n❌ Se encontraron archivos con errores:")
    for e in errors[:10]:
        print(e)

VERIFICACIÓN DEL FORMATO YOLO
Archivos analizados : 134
Archivos correctos  : 134
Archivos con errores: 0

✅ Todas las anotaciones cumplen el formato YOLO.


# 7. Verificación de imágenes corruptas

Como parte del proceso de limpieza del conjunto de datos, se verificó la integridad de todas las imágenes incluidas en los subconjuntos de entrenamiento, validación y prueba. Esta comprobación permite detectar archivos dañados o incompletos que podrían provocar errores durante el entrenamiento del modelo de detección de objetos.

In [ ]:
# ============================================================
# 7. VERIFICACIÓN DE IMÁGENES CORRUPTAS
# ------------------------------------------------------------
# Esta celda comprueba que todas las imágenes del dataset
# puedan abrirse correctamente. La detección de archivos
# corruptos forma parte del proceso de limpieza de datos.
# ============================================================

from PIL import Image
from pathlib import Path

subsets = ["train", "valid", "test"]

total_images = 0
corrupted_images = []

for subset in subsets:

    images_path = Path(extract_path) / subset / "images"

    for image_file in images_path.iterdir():

        if image_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
            continue

        total_images += 1

        try:
            img = Image.open(image_file)
            img.verify()

        except Exception:
            corrupted_images.append(str(image_file))

print("="*60)
print("VERIFICACIÓN DE IMÁGENES CORRUPTAS")
print("="*60)

print(f"Imágenes analizadas : {total_images}")
print(f"Imágenes corruptas  : {len(corrupted_images)}")

if len(corrupted_images) == 0:
    print("\n✅ No se detectaron imágenes corruptas.")
else:
    print("\n❌ Imágenes con problemas:")
    for img in corrupted_images[:10]:
        print(img)

VERIFICACIÓN DE IMÁGENES CORRUPTAS
Imágenes analizadas : 134
Imágenes corruptas  : 0

✅ No se detectaron imágenes corruptas.


### 7.1 Interpretación de los resultados

La verificación de integridad confirmó que todas las imágenes del conjunto de datos pueden abrirse correctamente y no presentan evidencias de corrupción o daños en su estructura. Este resultado garantiza que el dataset se encuentra en condiciones adecuadas para continuar con las etapas de preprocesamiento y entrenamiento del modelo YOLO.

# 8. Feature Engineering

El *Feature Engineering* corresponde al proceso de creación, transformación o selección de variables que permitan mejorar el desempeño de un modelo de aprendizaje automático. Sin embargo, en los modelos modernos de detección de objetos basados en redes neuronales convolucionales, como YOLO, este proceso difiere significativamente de los enfoques tradicionales.

En lugar de diseñar manualmente las características de entrada, la red aprende automáticamente representaciones jerárquicas a partir de las imágenes durante el proceso de entrenamiento. Por esta razón, el Feature Engineering en este proyecto se centra en la adecuada preparación del conjunto de datos, la correcta organización de las anotaciones en formato YOLO y la normalización de las coordenadas de las *bounding boxes*, más que en la construcción manual de nuevas variables.

# 9. Balanceamiento de clases

El balanceamiento de clases constituye una etapa importante durante la preparación de datos, ya que una distribución muy desigual puede ocasionar que el modelo aprenda con mayor facilidad las clases más representadas, reduciendo su capacidad para detectar correctamente las clases minoritarias.

En esta sección se analiza la distribución de objetos anotados por clase con el propósito de determinar si el conjunto de datos requiere la aplicación de técnicas adicionales de balanceamiento antes del entrenamiento del modelo YOLO.

In [ ]:
# ============================================================
# 9. BALANCEAMIENTO DE CLASES
# ------------------------------------------------------------
# Esta celda contabiliza el número de objetos anotados por
# clase en todo el conjunto de datos para evaluar si existe
# un desbalance significativo antes del entrenamiento.
# ============================================================

from collections import Counter

contador_clases = Counter()

subsets = ["train", "valid", "test"]

for subset in subsets:

    labels_path = Path(extract_path) / subset / "labels"

    for archivo in labels_path.glob("*.txt"):

        with open(archivo, "r") as f:

            for linea in f:

                datos = linea.strip().split()

                if len(datos) == 5:

                    clase = int(datos[0])

                    contador_clases[clase] += 1

print("="*60)
print("DISTRIBUCIÓN DE OBJETOS POR CLASE")
print("="*60)

for clase, cantidad in sorted(contador_clases.items()):
    print(f"Clase {clase}: {cantidad} objetos")

DISTRIBUCIÓN DE OBJETOS POR CLASE
Clase 0: 134 objetos


### 9.1 Interpretación de los resultados

El análisis evidenció que el conjunto de datos está conformado por una única clase, correspondiente al objeto de interés utilizado durante el entrenamiento del modelo YOLO. Debido a esta característica, no fue necesario aplicar técnicas de balanceamiento entre clases, ya que no existe una distribución desigual que pueda generar sesgos durante el proceso de aprendizaje.

En consecuencia, el entrenamiento se realizará sobre un conjunto de datos homogéneo, centrado exclusivamente en la detección de la clase objetivo.

# 10. Pipeline de preprocesamiento automatizado

El preprocesamiento automatizado corresponde al conjunto de actividades realizadas antes del entrenamiento del modelo, cuyo propósito es garantizar que el conjunto de datos cumpla con los requisitos técnicos establecidos por la arquitectura YOLO.

En este proyecto se implementó un pipeline reproducible que permitió organizar, validar y preparar automáticamente el dataset, asegurando la correcta correspondencia entre imágenes y anotaciones, la integridad de los archivos y el cumplimiento del formato requerido para el entrenamiento del modelo de detección de objetos.

In [ ]:
# ============================================================
# 10. PIPELINE DE PREPROCESAMIENTO AUTOMATIZADO
# ------------------------------------------------------------
# Esta celda resume las principales etapas realizadas durante
# la preparación del conjunto de datos antes del entrenamiento
# del modelo YOLO.
# ============================================================

pipeline = [
    "1. Carga del dataset",
    "2. Extracción del dataset",
    "3. Verificación de la estructura",
    "4. Validación imágenes ↔ etiquetas",
    "5. Verificación del formato YOLO",
    "6. Detección de imágenes corruptas",
    "7. Verificación del balance del dataset",
    "8. Dataset listo para entrenamiento"
]

print("="*65)
print("PIPELINE DE PREPROCESAMIENTO")
print("="*65)

for etapa in pipeline:
    print("✓", etapa)

print("\nProceso de preparación completado correctamente.")

PIPELINE DE PREPROCESAMIENTO
✓ 1. Carga del dataset
✓ 2. Extracción del dataset
✓ 3. Verificación de la estructura
✓ 4. Validación imágenes ↔ etiquetas
✓ 5. Verificación del formato YOLO
✓ 6. Detección de imágenes corruptas
✓ 7. Verificación del balance del dataset
✓ 8. Dataset listo para entrenamiento

Proceso de preparación completado correctamente.


### 10.1 Interpretación de los resultados

El pipeline de preprocesamiento permitió verificar de forma secuencial la calidad e integridad del conjunto de datos. Las diferentes etapas ejecutadas confirmaron que el dataset presenta una estructura adecuada, anotaciones válidas, imágenes sin corrupción y consistencia entre imágenes y etiquetas, garantizando que el conjunto de datos se encuentra preparado para iniciar el entrenamiento del modelo YOLO.

# 11. Data Augmentation

El aumento de datos (*Data Augmentation*) constituye una técnica ampliamente utilizada en visión por computadora para incrementar la variabilidad del conjunto de entrenamiento sin necesidad de capturar nuevas imágenes. Su objetivo es mejorar la capacidad de generalización del modelo, permitiendo que aprenda a reconocer el objeto de interés bajo diferentes condiciones de orientación, iluminación y calidad visual.

En este proyecto, las técnicas de aumento de datos fueron implementadas mediante la plataforma Roboflow durante la generación del dataset utilizado para el entrenamiento del modelo YOLO.

In [ ]:
# ============================================================
# 11. DATA AUGMENTATION
# ------------------------------------------------------------
# Resumen de las técnicas de aumento de datos aplicadas en
# Roboflow durante la generación del dataset.
# ============================================================

print("="*65)
print("TÉCNICAS DE DATA AUGMENTATION APLICADAS")
print("="*65)

print("Preprocessing")
print("• Auto-Orient: Applied")
print("• Resize: Fit (black edges) 640x640")

print("\nData Augmentation")
print("• Outputs per training example: 3")
print("• Rotation: -15° a +15°")
print("• Brightness: -25% a +25%")
print("• Blur: hasta 2.5 px")

TÉCNICAS DE DATA AUGMENTATION APLICADAS
Preprocessing
• Auto-Orient: Applied
• Resize: Fit (black edges) 640x640

Data Augmentation
• Outputs per training example: 3
• Rotation: -15° a +15°
• Brightness: -25% a +25%
• Blur: hasta 2.5 px


# 12. Dataset listo para entrenamiento

Una vez finalizadas las etapas de limpieza, validación, preparación y aumento de datos, se realizó una verificación final del conjunto de datos con el propósito de confirmar que cumple con las condiciones necesarias para iniciar el entrenamiento del modelo YOLO.

Esta validación integra los resultados obtenidos durante las etapas anteriores y resume el estado final del dataset utilizado en el presente proyecto.

In [ ]:
# ============================================================
# 12. DATASET LISTO PARA ENTRENAMIENTO
# ------------------------------------------------------------
# Esta celda presenta un resumen del estado final del conjunto
# de datos después de completar todas las etapas de preparación.
# ============================================================

print("="*65)
print("RESUMEN FINAL DEL DATASET")
print("="*65)

print(f"Total de imágenes           : 134")
print(f"Total de clases             : 1")
print(f"Train                       : 117 imágenes")
print(f"Validation                  : 11 imágenes")
print(f"Test                        : 6 imágenes")

print("\nVerificaciones realizadas")

print("✅ Estructura del dataset verificada")
print("✅ Correspondencia imágenes-etiquetas")
print("✅ Formato YOLO validado")
print("✅ Sin imágenes corruptas")
print("✅ Dataset balanceado para su propósito")
print("✅ Data Augmentation aplicado")
print("✅ Dataset preparado para entrenamiento")

print("\nEstado final: DATASET LISTO PARA ENTRENAR EL MODELO YOLO")

RESUMEN FINAL DEL DATASET
Total de imágenes           : 134
Total de clases             : 1
Train                       : 117 imágenes
Validation                  : 11 imágenes
Test                        : 6 imágenes

Verificaciones realizadas
✅ Estructura del dataset verificada
✅ Correspondencia imágenes-etiquetas
✅ Formato YOLO validado
✅ Sin imágenes corruptas
✅ Dataset balanceado para su propósito
✅ Data Augmentation aplicado
✅ Dataset preparado para entrenamiento

Estado final: DATASET LISTO PARA ENTRENAR EL MODELO YOLO


### 12.1 Interpretación de los resultados

La validación final confirmó que el conjunto de datos cumple con todos los requisitos necesarios para el entrenamiento del modelo YOLO. La estructura del dataset, la integridad de las imágenes, la consistencia de las anotaciones y las técnicas de preprocesamiento aplicadas permiten concluir que el conjunto de datos presenta las condiciones adecuadas para el desarrollo del modelo de detección de objetos.

# 13. Conclusiones

- El proceso de preparación y procesamiento permitió verificar la calidad estructural del conjunto de datos, garantizando la consistencia entre imágenes y anotaciones antes del entrenamiento del modelo YOLO.

- La aplicación de técnicas de preprocesamiento y aumento de datos mediante Roboflow incrementó la calidad y variabilidad del dataset, favoreciendo la capacidad de generalización del modelo frente a diferentes condiciones de captura.

- La validación integral realizada durante esta etapa permitió confirmar que el conjunto de datos se encuentra correctamente organizado, documentado y preparado para iniciar el entrenamiento del modelo de detección de objetos.